In [1]:
# ---------------------------------------------------------------------
#  Imports
# ---------------------------------------------------------------------
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import country_converter as coco
import numpy as np
import time
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox
import re
from uncertainties import unumpy as unp
from uncertainties import ufloat, nominal_value, std_dev


# ---------------------------------------------------------------------
#  File paths
# ---------------------------------------------------------------------
file_path_main = "../../results/scale_up/Scale_up_output_MS.pkl"
file_path_percent = "../../results/scale_up/Scale_up_PERCENT_INDOOR_VITAL_MS.pkl"
file_path_cr_man = "../../results/scale_up/Scale_up_CR_MAN_MS.pkl"
file_path_cr_repur = "../../results/scale_up/Scale_up_CR_REPUR_MS.pkl"
file_path_coalbag = "../../results/scale_up/Scale_up_COALBAG_MS.pkl"
file_path_cr_stock = "../../results/scale_up/Scale_up_CR_STOCK.pkl"
file_path_essential_workers_country = "../../results/essential_workers/EssentialWorkersByCountry.csv"

# ---------------------------------------------------------------------
#  Functions Import
# ---------------------------------------------------------------------
def get_series(df, region):
    row = df.loc[region]
    row = row.iloc[2:]
    time = np.arange(len(row))
    values = np.array([nominal_value(v) for v in row])
    errors = np.array([std_dev(v) for v in row])
    return time, values, errors

def get_indoor_vital_bounds(df, region):
    row = df.loc[region]
    indoor_vital_upper = row[1]
    indoor_vital_lower = row[0]
    return indoor_vital_lower, indoor_vital_upper

CADRPP = 100 # L/s
UNRegion_list = ['Australia and New Zealand',
 'Caribbean',
 'Central America',
 'Central Asia',
 'Eastern Africa',
 'Eastern Asia',
 'Eastern Europe',
 'Melanesia',
 'Micronesia',
 'Middle Africa',
 'Northern Africa',
 'Northern America',
 'Northern Europe',
 'Polynesia',
 'South America',
 'South-eastern Asia',
 'Southern Africa',
 'Southern Asia',
 'Southern Europe',
 'Western Africa',
 'Western Asia',
 'Western Europe']

# ---------------------------------------------------------------------
#  Load CSVs (main airflow + percentage + subtypes)
# ---------------------------------------------------------------------
def load_dataset(path):
    df = pd.read_pickle(path)
    week_cols = df.columns[3:]
    df_filtered = df.loc[UNRegion_list]
    df_nominal = df_filtered.map(
        lambda x: unp.nominal_values(x) if hasattr(x, "nominal_value") else x
    )   
    df_nominal = df_nominal.drop(columns=[df.columns[0], df.columns[1]]) 
    return df_nominal, week_cols

# Load datasets
df_main, week_cols = load_dataset(file_path_main)
df_percent, _ = load_dataset(file_path_percent)
df_cr_man, _ = load_dataset(file_path_cr_man)
df_cr_repur, _ = load_dataset(file_path_cr_repur)
df_coalbag, _ = load_dataset(file_path_coalbag)
df_cr_stock, _ = load_dataset(file_path_cr_stock)

# Load indoor vital lower/upper bounds
df_indoor_vital = pd.read_pickle(file_path_main)
df_indoor_vital = df_indoor_vital.loc[UNRegion_list]
df_indoor_vital = df_indoor_vital.iloc[:,:2]
# print(df_indoor_vital)
indoor_vital_lower_dict={}
indoor_vital_upper_dict = {}
for region in UNRegion_list:
    row = df_indoor_vital.loc[region]
    indoor_vital_upper_dict[region] = CADRPP*row[1]
    indoor_vital_lower_dict[region] = CADRPP*row[0]



In [2]:
# ---------------------------------------------------------------------
#  Widgets
# ---------------------------------------------------------------------
map_choice = widgets.Dropdown(
    options=[
        "ALL",
        "Indoor Vital Coverage %",
        "CR Box Manufacturing",
        "CR Box Repurposing",
        "Coalbaghouse",
        "CR Box Stock",
    ],
    value="ALL",
    description="Type:",
    style={"description_width": "initial"},
)

region_filter = widgets.Dropdown(
    options=["All"] + sorted(UNRegion_list),
    value="All",
    description="UN Region:",
    style={"description_width": "initial"},
)

region_select = widgets.SelectMultiple(
    options=sorted(UNRegion_list),
    value=["Eastern Asia"],
    description="Regions:",
    layout=widgets.Layout(width="50%", height="150px"),
)

week_index = widgets.IntSlider(
    value=1,
    min=1,
    max=len(week_cols),
    step=1,
    description="Week:",
    layout=widgets.Layout(width="80%"),
)

play_button = widgets.Button(
    description="▶ Play",
    tooltip="Auto-play weeks",
    button_style="success",
    layout=widgets.Layout(width="100px", height="35px"),
)

out = widgets.Output()

# ---------------------------------------------------------------------
#  CADRPP input + confirm button
# ---------------------------------------------------------------------
CADRPP_input = widgets.FloatText(
    value=100.0,
    description="CADRPP (L/s per person):",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="280px")
)

confirm_CADRPP_button = widgets.Button(
    description="Enter",
    tooltip="Confirm CADRPP value",
    button_style="info",
    layout=widgets.Layout(width="80px", height="35px"),
)

def confirm_CADRPP_action(b):
    global CADRPP, indoor_vital_lower_dict, indoor_vital_upper_dict
    CADRPP = CADRPP_input.value
    print(f"[INFO] CADRPP confirmed as {CADRPP} L/s per person")
    # Update bounds dynamically
    for region in UNRegion_list:
        row = df_indoor_vital.loc[region]
        indoor_vital_upper_dict[region] = CADRPP * row[1]
        indoor_vital_lower_dict[region] = CADRPP * row[0]
    update_plots()

confirm_CADRPP_button.on_click(confirm_CADRPP_action)
CADRPP_box = HBox([CADRPP_input, confirm_CADRPP_button])

# ---------------------------------------------------------------------
#  Plot update
# ---------------------------------------------------------------------
def update_plots(change=None):
    week = str(week_index.value)
    map_val = map_choice.value
    with out:
        clear_output(wait=True)

        # ---------- Choose dataset ----------
        if map_val == "Indoor Vital Coverage %":
            df_map = df_percent.copy()
            vmin, vmax = 0, 100
            color_scale = [(0, "white"), (1, "orange")]
            title_text = f"Indoor Vital Coverage (%) by Country (Week {week})"
        elif map_val == "CR Box Manufacturing":
            df_map = df_cr_man.copy(); color_scale = [(0, "white"), (1, "orange")]
            vmin, vmax = df_map.min().min(), df_map.max().max()
            title_text = f"CR Box Manufacturing (Week {week})"
        elif map_val == "CR Box Repurposing":
            df_map = df_cr_repur.copy(); color_scale = [(0, "white"), (1, "orange")]
            vmin, vmax = df_map.min().min(), df_map.max().max()
            title_text = f"CR Box Repurposing (Week {week})"
        elif map_val == "Coalbaghouse":
            df_map = df_coalbag.copy(); color_scale = [(0, "white"), (1, "orange")]
            vmin, vmax = df_map.min().min(), df_map.max().max()
            title_text = f"Coalbaghouse (Week {week})"
        elif map_val == "CR Box Stock":
            df_map = df_cr_stock.copy(); color_scale = [(0, "white"), (1, "orange")]
            vmin, vmax = df_map.min().min(), df_map.max().max()
            title_text = f"CR Box Stock (Week {week})"
        else:
            df_map = df_main.copy(); color_scale = [(0, "white"), (1, "darkgreen")]
            vmin, vmax = df_map.min().min(), df_map.max().max()
            title_text = f"Total Air Flow by Country (up to Week {week})"

        # ---------- Choropleth ----------
        country_values = []
        for region_name, row in df_map.iterrows():
            if region_name not in region_to_iso: continue
            try: value = row[int(week)]
            except Exception: continue
            for iso3 in region_to_iso[region_name]:
                country_values.append({"ISO3": iso3, "Region": region_name, week: value})
        choropleth_df = pd.DataFrame(country_values)
        if not choropleth_df.empty:
            fig1 = px.choropleth(
                choropleth_df, locations="ISO3", locationmode="ISO-3",
                color=week, hover_name="Region",
                color_continuous_scale=color_scale, range_color=[vmin, vmax],
                title=title_text)
            fig1.update_layout(margin=dict(l=0, r=0, t=50, b=0))
            fig1.show()
        else:
            print("[⚠️] No data found — check mapping or week index.")

        # ---------- Controls ----------
        display(HBox([week_index, play_button]))

        # ---------- Second graph: Airflow + Indoor Vital range ----------
        fig, ax = plt.subplots(figsize=(10, 6))
        df_used = {
            "ALL": df_main, "CR Box Manufacturing": df_cr_man,
            "CR Box Repurposing": df_cr_repur, "Coalbaghouse": df_coalbag,
            "CR Box Stock": df_cr_stock, "Indoor Vital Coverage %": df_percent
        }.get(map_val, df_main)

        # --- Plot regions ---
        for region in region_select.value:
            if region not in df_used.index: continue
            mean_series = df_used.loc[region].astype(float)
            t = np.arange(len(mean_series))
            ax.plot(t, mean_series, label=f"{region} ({map_val})", linewidth=2)

            # --- Only for non-percentage types ---
            if map_val != "Indoor Vital Coverage %":
                lower = indoor_vital_lower_dict[region]
                upper = indoor_vital_upper_dict[region]

                # Indoor Vital range bands
                ax.axhspan(lower, upper, color="green", alpha=0.1,
                           label="Indoor Vital Range" if region == region_select.value[0] else "")
                ax.hlines([lower, upper], xmin=0, xmax=len(mean_series)-1,
                          colors=["lime", "orange"], linestyles="--",
                          label="Indoor Vital Lower" if region == region_select.value[0] else None)
                
                # --- Detect intersections with bounds ---
                y = mean_series.values
                weeks = np.arange(len(y))

                # Lower-bound crossings
                low_cross = np.where((y[:-1] < lower) & (y[1:] >= lower) |
                                     (y[:-1] > lower) & (y[1:] <= lower))[0]
                # Upper-bound crossings
                up_cross = np.where((y[:-1] < upper) & (y[1:] >= upper) |
                                    (y[:-1] > upper) & (y[1:] <= upper))[0]

                for idx in np.concatenate([low_cross, up_cross]):
                    w = weeks[idx+1]
                    val = y[idx+1]
                    ax.scatter(w, val, color="red", s=40, zorder=5)
                    ax.text(w, val, f" Week {w}\n{val:,.0f}",
                            color="red", fontsize=8, ha="left", va="bottom")

        ax.set_title(f"{map_val} vs Indoor Vital Range (CADRPP = {CADRPP:.1f} L/s)")
        ax.set_xlabel("Weeks")
        ax.set_ylabel("Air Flow (L/s)")
        ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
        plt.tight_layout()
        plt.show()

# ---------------------------------------------------------------------
#  Play logic
# ---------------------------------------------------------------------
def play_clicked(b):
    for w in range(1, len(week_cols) + 1):
        week_index.value = w
        time.sleep(0.5)

play_button.on_click(play_clicked)
week_index.observe(update_plots, names="value")
map_choice.observe(update_plots, names="value")
region_select.observe(update_plots, names="value")

# ---------------------------------------------------------------------
#  Layout + Style
# ---------------------------------------------------------------------
custom_style = """
<style>
#custom-container {
    background-color: #fffacd;
    padding: 15px;
    border-radius: 10px;
    margin-top: 10px;
    margin-bottom: 20px;
}
#custom-banner {
    width: 100%;
    background-color: #006400;
    color: white;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    padding: 10px;
    border-radius: 6px;
    margin-bottom: 15px;
}
</style>
"""
display(HTML(custom_style))

container = VBox([
    widgets.HTML('<div id="custom-banner">ALLFED</div>'),
    HBox([region_filter, map_choice, CADRPP_box]),
    region_select,
    out
])
container.layout = widgets.Layout(
    width="100%",
    padding="15px",
    border="solid 2px #006400",
    border_radius="10px",
    background_color="#fffacd",
)

display(container)
try:
    region_to_iso
except NameError:
    cc = coco.CountryConverter()
    region_to_iso = {}
    for region in UNRegion_list:
        matches = cc.data.loc[cc.data["UNregion"] == region, "ISO3"].tolist()
        region_to_iso[region] = matches
    print("[INFO] region_to_iso mapping rebuilt automatically.")

update_plots()


[INFO] region_to_iso mapping rebuilt automatically.


In [14]:
import country_converter as coco
cc = coco.CountryConverter()

print("=== Unique values in cc.data['UNregion'] ===")
print(cc.data['UNregion'].dropna().unique())
print("\n=== Unique values in cc.data['continent'] ===")
print(cc.data['continent'].dropna().unique())


=== Unique values in cc.data['UNregion'] ===
['Southern Asia' 'Northern Europe' 'Southern Europe' 'Northern Africa'
 'Polynesia' 'Middle Africa' 'Caribbean' 'Antarctica' 'South America'
 'Western Asia' 'Australia and New Zealand' 'Western Europe'
 'Eastern Europe' 'Central America' 'Western Africa' 'Northern America'
 'Southern Africa' 'Eastern Africa' 'South-eastern Asia' 'Eastern Asia'
 'Melanesia' 'Micronesia' 'Central Asia']

=== Unique values in cc.data['continent'] ===
['Asia' 'Europe' 'Africa' 'Oceania' 'America' 'Antarctica']
